# Appendix E: Data Dictionary and Schema

Inspect and validate the NRG machine-readable data contract.

**Level:** Beginner to intermediate  |  **Time:** 45 minutes

All data and organisations are fictional.


In [ ]:
from pathlib import Path
import sys,pandas as pd,matplotlib.pyplot as plt
root=Path.cwd().parents[1];sys.path.insert(0,str(root/'src'))
from datasciencebook.schema_contracts import load_dictionary,load_schema,table_contract,validate_dictionary
rows=load_dictionary(root/'data/dictionaries/nrg_data_dictionary.csv');schema=load_schema(root/'data/dictionaries/nrg_schema.json')
print('Schema:',schema['schema_version'],'Fictional:',schema['fictional'])


Schema: 1.0.0 Fictional: True


In [ ]:
report=validate_dictionary(rows)
print(report)
print(table_contract(rows,'forecasts'))


{'rows': 38, 'tables': ['forecasts', 'inventory_snapshots', 'locations', 'planning_decisions', 'products', 'weekly_demand'], 'errors': [], 'valid': True}
{'table': 'forecasts', 'columns': ['forecast_origin', 'target_week', 'product_id', 'market_id', 'model_version', 'p50_cases', 'p90_cases'], 'required': ['forecast_origin', 'target_week', 'product_id', 'market_id', 'model_version', 'p50_cases', 'p90_cases'], 'keys': ['forecast_origin', 'target_week', 'product_id', 'market_id']}


In [ ]:
products=pd.DataFrame({'product_id':['P001','P002'],'product_name':['Jakarta Noodles','Gayo Coffee'],'category':['noodles','coffee'],'shelf_life_days':[270,365],'case_cost_usd':[11.5,28.0]})
demand=pd.DataFrame({'week_start':['2026-08-03','2026-08-03'],'product_id':['P001','P002'],'market_id':['L003','L003'],'demand_cases':[120,80],'sales_cases':[110,80],'promotion_flag':[True,False],'stockout_days':[1,0]})
joined=demand.merge(products,on='product_id',validate='many_to_one')
print('Rows preserved:',len(joined)==len(demand));print('Demand cases:',joined['demand_cases'].sum())


Rows preserved: True
Demand cases: 200


In [ ]:
forecasts=pd.DataFrame({'product_id':['P001','P002'],'p50_cases':[125,82],'p90_cases':[150,96]})
assert (forecasts['p90_cases']>=forecasts['p50_cases']).all()
forecasts['uncertainty_width']=forecasts['p90_cases']-forecasts['p50_cases']
print(forecasts.to_dict('records'))


[{'product_id': 'P001', 'p50_cases': 125, 'p90_cases': 150, 'uncertainty_width': 25}, {'product_id': 'P002', 'p50_cases': 82, 'p90_cases': 96, 'uncertainty_width': 14}]


In [ ]:
dictionary=pd.DataFrame(rows);counts=dictionary.groupby('table').size().sort_values()
fig,axes=plt.subplots(1,2,figsize=(10,4));counts.plot.barh(ax=axes[0],color='#4f8a63');axes[0].set(title='Fields per table',xlabel='Fields',ylabel='');joined.plot.bar(x='product_name',y=['sales_cases','demand_cases'],ax=axes[1],color=['#795548','#d7a84b']);axes[1].set(title='Sales and demand',xlabel='',ylabel='Cases');fig.tight_layout();plt.show()


## Interpretation

The dictionary is executable metadata. It exposes table grain, field requirements, units, and rules before analysis begins. Join validation preserves row grain, while forecast checks protect quantile ordering.


In [ ]:
# Practice: add one invalid forecast row and write a check that rejects it.
